<a href="https://colab.research.google.com/github/duane-edgington/perch-hoplite/blob/main/MBARI_perch_phase1_embed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# -*- coding: utf-8 -*-
"""MBARI_perch_phase1_embed.ipynb

Perch Hoplite Phase 1 — Build Embedding Database
MBARI Marine Sound Classification Pipeline
================================================

PURPOSE
-------
This notebook reads MARS hydrophone audio files (pre-staged as a zip in
Google Drive by prepare_audio_for_colab.sh on your Mac), embeds them with
the Perch V2 model, and saves the resulting Hoplite database back to
Google Drive for transfer to spark-ae0e.

WORKFLOW
--------
1. Run prepare_audio_for_colab.sh on your Mac to zip audio and upload to Drive
2. Open this notebook in Colab with a GPU runtime
3. Run all cells in order
4. When complete, the database (hoplite.sqlite + usearch.index) will be in
   your Google Drive at MBARI_perch/db/<dataset_name>/
5. Download those two files and upload to spark-ae0e:
   /mnt/PAM_Analysis/duane_scratch/perch_hoplite/db/<dataset_name>/

RUNTIME REQUIREMENT
-------------------
This notebook REQUIRES a GPU runtime. In Colab:
  Runtime > Change runtime type > Hardware accelerator = GPU (T4 or better)

The Perch V2 model uses XLA-compiled kernels that cannot run on CPU-only
runtimes. A T4 GPU is sufficient and available on free Colab tier.
"""


In [ ]:
# ============================================================
# CELL 1 — Install perch-hoplite and TensorFlow
# ============================================================
# @title Step 1: Install dependencies
# @markdown Run once per session. After installation you MUST restart
# @markdown the runtime: Runtime > Restart session.
# @markdown Then continue from Cell 2 (do NOT re-run this cell).

# Install perch-hoplite from GitHub source
# (PyPI version 1.0.1 has a known XLA incompatibility with newer GPUs;
#  GitHub HEAD is the safest option)
get_ipython().system('pip install -q git+https://github.com/google-research/perch-hoplite.git')

# Perch V2 requires TF >= 2.20rc for best GPU compatibility.
# This replaces the standard tensorflow package.
get_ipython().system('pip install -q tensorflow[and-cuda]~=2.20.0rc0')

print("\n*** Installation complete ***")
print(">>> RESTART THE RUNTIME NOW: Runtime > Restart session")
print(">>> Then run from Cell 2 onwards. Do NOT re-run this cell.")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 145.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 135.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ============================================================
# CELL 2 — Imports
# ============================================================
# @title Step 2: Imports
# @markdown Run after restarting the runtime.

import os
import shutil
import zipfile
from pathlib import Path

from etils import epath
from ml_collections import config_dict
import numpy as np

from perch_hoplite.agile import colab_utils
from perch_hoplite.agile import embed
from perch_hoplite.agile import source_info
from perch_hoplite.db import brutalism
from perch_hoplite.db import interface
from perch_hoplite.zoo import taxonomy_model_tf

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow {tf.__version__}")
print(f"GPUs visible: {len(gpus)}")
if gpus:
    for g in gpus:
        details = tf.config.experimental.get_device_details(g)
        print(f"  GPU: {details.get('device_name', g.name)}, "
              f"compute capability: {details.get('compute_capability', 'unknown')}")
else:
    print("WARNING: No GPU detected. Perch V2 requires a GPU runtime.")
    print("         Go to Runtime > Change runtime type > GPU")


TensorFlow 2.20.0
GPUs visible: 1
  GPU: NVIDIA A100-SXM4-80GB, compute capability: (8, 0)


In [ ]:
# ============================================================
# CELL 3 — Mount Google Drive
# ============================================================
# @title Step 3: Mount Google Drive
# @markdown You will be prompted to authorise access.

from google.colab import drive
drive.mount('/content/drive')

GDRIVE_BASE = '/content/drive/My Drive'
print(f"Google Drive mounted at: {GDRIVE_BASE}")
print(f"Contents of MBARI_perch (if it exists):")
mbari_dir = Path(GDRIVE_BASE) / 'MBARI_perch'
if mbari_dir.exists():
    for item in sorted(mbari_dir.iterdir()):
        print(f"  {item.name}/")
else:
    print("  (MBARI_perch folder not found yet — it will be created below)")


Mounted at /content/drive
Google Drive mounted at: /content/drive/My Drive
Contents of MBARI_perch (if it exists):
  audio/
  db/


In [ ]:
# ============================================================
# CELL 4 — Configuration
# ============================================================
# @title Step 4: Configuration
# @markdown Edit the values in this cell to match what
# @markdown prepare_audio_for_colab.sh produced on your Mac.
# @markdown The script writes a colab_config_*.txt file to your
# @markdown Drive with these values pre-filled.

# ---- Input: zip file path relative to Google Drive root ----
# This is the zip created by prepare_audio_for_colab.sh
GDRIVE_AUDIO_ZIP = 'MBARI_perch/audio/MARS_20180401_20180401_32kHz.zip'  # @param {type:'string'}

# ---- Dataset identity ----
# Must be unique per embedding run. Used as the DB folder name and
# as the Hoplite project/dataset name inside the database.
DATASET_NAME = 'MARS_20180401_20180401_32kHz'  # @param {type:'string'}

# ---- Model choice ----
# perch_v2     : 32 kHz, 5-second windows — matches resampled_32kHz audio
# multispecies_whale : 24 kHz, 5-second windows — use with resampled_24kHz audio
MODEL_CHOICE = 'perch_v2'  # @param ['perch_v2', 'multispecies_whale', 'humpback', 'surfperch']

# ---- Sharding ----
# Split each audio file into overlapping chunks before embedding.
# 75 seconds is a good balance of memory use vs. overhead.
SHARD_LEN_S = 75  # @param {type:'number'}

# ---- Output: where to write the database in Google Drive ----
# The finished DB will be at GDRIVE_BASE/GDRIVE_DB_DIR/<DATASET_NAME>/
GDRIVE_DB_DIR = 'MBARI_perch/db'  # @param {type:'string'}

# ---- Local working directory inside Colab VM ----
# Audio is extracted here before embedding. Colab VMs have ~100 GB local disk.
LOCAL_AUDIO_DIR = f'/tmp/mbari_audio/{DATASET_NAME}'
LOCAL_DB_DIR    = f'/tmp/mbari_db/{DATASET_NAME}'

# ---- Derived paths ----
zip_full_path  = Path(GDRIVE_BASE) / GDRIVE_AUDIO_ZIP
db_output_path = Path(GDRIVE_BASE) / GDRIVE_DB_DIR / DATASET_NAME

print("=== Configuration ===")
print(f"  Audio zip  : {zip_full_path}")
print(f"  Zip exists : {zip_full_path.exists()}")
print(f"  Dataset    : {DATASET_NAME}")
print(f"  Model      : {MODEL_CHOICE}")
print(f"  Shard len  : {SHARD_LEN_S}s")
print(f"  Local audio: {LOCAL_AUDIO_DIR}")
print(f"  Local DB   : {LOCAL_DB_DIR}")
print(f"  Drive DB   : {db_output_path}")


=== Configuration ===
  Audio zip  : /content/drive/My Drive/MBARI_perch/audio/MARS_20180401_20180401_32kHz.zip
  Zip exists : True
  Dataset    : MARS_20180401_20180401_32kHz
  Model      : perch_v2
  Shard len  : 75s
  Local audio: /tmp/mbari_audio/MARS_20180401_20180401_32kHz
  Local DB   : /tmp/mbari_db/MARS_20180401_20180401_32kHz
  Drive DB   : /content/drive/My Drive/MBARI_perch/db/MARS_20180401_20180401_32kHz


In [ ]:
# ============================================================
# CELL 5 — Extract audio from zip
# ============================================================
# @title Step 5: Extract audio files from zip
# @markdown Extracts the zip from Google Drive to the local Colab VM disk.
# @markdown This is much faster than reading from Drive during embedding.

if not zip_full_path.exists():
    raise FileNotFoundError(
        f"Zip not found: {zip_full_path}\n"
        f"Run prepare_audio_for_colab.sh on your Mac first, "
        f"then wait for Google Drive to sync."
    )

print(f"Extracting {zip_full_path.name} ...")
os.makedirs(LOCAL_AUDIO_DIR, exist_ok=True)

with zipfile.ZipFile(zip_full_path, 'r') as zf:
    members = zf.namelist()
    print(f"  Zip contains {len(members)} files.")
    zf.extractall(LOCAL_AUDIO_DIR)

wav_files = sorted(Path(LOCAL_AUDIO_DIR).glob('*.wav'))
print(f"  Extracted {len(wav_files)} .wav files.")
if wav_files:
    print(f"  First file : {wav_files[0].name}")
    print(f"  Last file  : {wav_files[-1].name}")

# Sanity check: show disk usage
result = os.popen(f'du -sh {LOCAL_AUDIO_DIR}').read().strip()
print(f"  Disk usage : {result}")


Extracting MARS_20180401_20180401_32kHz.zip ...
  Zip contains 144 files.
  Extracted 144 .wav files.
  First file : MARS_20180401_000914_resampled_32kHz.wav
  Last file  : MARS_20180401_235911_resampled_32kHz.wav
  Disk usage : 5.2G	/tmp/mbari_audio/MARS_20180401_20180401_32kHz


In [ ]:
# ============================================================
# CELL 6 — Initialise database
# ============================================================
# @title Step 6: Initialise Hoplite database
# @markdown Creates the SQLite + USearch database on local Colab disk.
# @markdown If a database already exists at LOCAL_DB_DIR from a previous
# @markdown run, set DROP_EXISTING = True to start fresh.

DROP_EXISTING = False  # @param {type:'boolean'}

os.makedirs(LOCAL_DB_DIR, exist_ok=True)
db_path_epath = epath.Path(LOCAL_DB_DIR)

# Set up audio source config pointing to local extracted audio
audio_glob = source_info.AudioSourceConfig(
    dataset_name=DATASET_NAME,
    base_path=LOCAL_AUDIO_DIR,
    file_glob='*.wav',
    min_audio_len_s=1.0,
    target_sample_rate_hz=-1,          # use model's native sample rate
    shard_len_s=float(SHARD_LEN_S),
)

configs = colab_utils.load_configs(
    source_info.AudioSources((audio_glob,)),
    db_path=LOCAL_DB_DIR,
    model_config_key=MODEL_CHOICE,
    db_key='sqlite_usearch',
)

db = configs.db_config.load_db()
num_existing = db.count_embeddings()
print(f"Database at  : {LOCAL_DB_DIR}")
print(f"Existing embeddings: {num_existing}")

if num_existing > 0:
    if DROP_EXISTING:
        print("DROP_EXISTING=True — deleting existing embeddings...")
        for fp in db_path_epath.glob('hoplite.sqlite*'):
            fp.unlink()
        (db_path_epath / 'usearch.index').unlink()
        db = configs.db_config.load_db()
        print("Database reset.")
    else:
        print("Existing embeddings found. Set DROP_EXISTING=True to start fresh.")
        print("Embedding will APPEND to existing database.")


Database at  : /tmp/mbari_db/MARS_20180401_20180401_32kHz
Existing embeddings: 0


In [ ]:
# ============================================================
# CELL 7 — Run embedding
# ============================================================
# @title Step 7: Run embedding
# @markdown This is the main compute step. Runtime depends on:
# @markdown   - Number of audio files
# @markdown   - GPU type (T4 ≈ 2-5 sec/file, A100 ≈ 0.5-1 sec/file)
# @markdown   - Shard length
# @markdown
# @markdown For 150 files (≈25 hours audio) on a T4: expect 15-30 minutes.
# @markdown For 4,320 files (full April 2018): expect 6-18 hours.
# @markdown Use Runtime > Run all, then close your laptop — Colab keeps running.
# @markdown
# @markdown Progress is logged every file. If the session disconnects, re-run
# @markdown this cell — it will skip already-embedded files.

import time
start_time = time.time()

print(f"Starting embedding of dataset: {DATASET_NAME}")
print(f"Model: {MODEL_CHOICE}, Shard: {SHARD_LEN_S}s")
print(f"Files to embed: {len(wav_files)}")
print("-" * 60)

worker = embed.EmbedWorker(
    audio_sources=configs.audio_sources_config,
    db=db,
    model_config=configs.model_config,
)

worker.process_all(target_dataset_name=audio_glob.dataset_name)

elapsed = time.time() - start_time
total_embeddings = db.count_embeddings()

print("-" * 60)
print(f"Embedding complete.")
print(f"  Total embeddings : {total_embeddings:,}")
print(f"  Elapsed time     : {elapsed/60:.1f} minutes")
if len(wav_files) > 0:
    print(f"  Avg per file     : {elapsed/len(wav_files):.1f} seconds")


Starting embedding of dataset: MARS_20180401_20180401_32kHz
Model: perch_v2, Shard: 75s
Files to embed: 144
------------------------------------------------------------



Adding deployments...


100%|██████████| 144/144 [00:00<00:00, 2699.58it/s]



Adding recordings...


100%|██████████| 144/144 [00:00<00:00, 6019.51it/s]



Adding annotations...

Embedding audio...


100%|██████████| 144/144 [01:20<00:00,  1.80it/s]

------------------------------------------------------------
Embedding complete.
  Total embeddings : 17,280
  Elapsed time     : 1.5 minutes
  Avg per file     : 0.6 seconds


In [ ]:
# ============================================================
# CELL 8 — Verify database
# ============================================================
# @title Step 8: Verify database
# @markdown Runs a nearest-neighbour sanity check and prints per-project stats.

print("=== Database verification ===")
print()

# Per-project stats
for project in db.get_all_projects():
    window_ids = db.match_window_ids(
        deployments_filter=config_dict.create(eq=dict(project=project))
    )
    print(f"Project : {project}")
    print(f"  Embeddings: {len(window_ids):,}")
    print()

# Nearest-neighbour search sanity check
print("Running nearest-neighbour sanity check...")
try:
    q = db.get_embedding(db.match_window_ids(limit=1)[0])
    results, scores = brutalism.brute_search(
        db, query_embedding=q, search_list_size=32, score_fn=np.dot
    )
    print(f"  Search returned {len(results)} results — database is functional.")
    print(f"  Top score: {scores[0]:.4f} (should be 1.0 — self-match)")
except Exception as e:
    print(f"  WARNING: Sanity check failed: {e}")
    print("  This may be normal if only a small number of embeddings were created.")

# List DB files and sizes
print()
print("Database files:")
for f in sorted(Path(LOCAL_DB_DIR).iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:30s}  {size_mb:.1f} MB")


=== Database verification ===

Project : MARS_20180401_20180401_32kHz
  Embeddings: 17,280

Running nearest-neighbour sanity check...
      dtype=float16), array([-0.0441  ,  0.01047 , -0.02995 , ..., -0.01901 , -0.009636,
        0.1353  ], dtype=float16), array([-0.0943  ,  0.001824,  0.01831 , ...,  0.007122,  0.01217 ,
        0.1509  ], dtype=float16), array([-0.0709  ,  0.009544, -0.03442 , ...,  0.03156 ,  0.006516,
        0.2725  ], dtype=float16), array([-0.02931 ,  0.02237 ,  0.012276, ..., -0.00473 , -0.0404  ,
        0.1259  ], dtype=float16), array([-0.0698  ,  0.007427, -0.0702  , ..., -0.007736, -0.01082 ,
        0.09    ], dtype=float16), array([-0.05414 ,  0.02711 , -0.001637, ..., -0.0351  , -0.0468  ,
        0.1208  ], dtype=float16), array([-0.08765,  0.03333,  0.01952, ...,  0.02385, -0.01646,  0.1946 ],
      dtype=float16), array([-0.1271  ,  0.006157, -0.01852 , ...,  0.022   , -0.00573 ,
        0.126   ], dtype=float16), array([-0.063   ,  0.01799 ,  0.043

In [ ]:
# @title Step 8b: Force SQLite WAL checkpoint
# @markdown Flushes the Write-Ahead Log into the main database file
# @markdown so that hoplite.sqlite contains all embeddings before we zip.

import sqlite3 as sqlite_builtin

db_file = os.path.join(LOCAL_DB_DIR, 'hoplite.sqlite')
conn = sqlite_builtin.connect(db_file)
conn.execute("PRAGMA wal_checkpoint(TRUNCATE);")
conn.commit()
conn.close()

# Verify the main file now has data
import os
for f in sorted(Path(LOCAL_DB_DIR).iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:35s}  {size_mb:.1f} MB")

print()
print("hoplite.sqlite should now be > 0 MB")

  hoplite.sqlite                       1.2 MB
  hoplite.sqlite-shm                   0.0 MB
  hoplite.sqlite-wal                   0.0 MB
  usearch.index                        53.1 MB

hoplite.sqlite should now be > 0 MB


In [ ]:
# @title Step 9: Zip database and save to Google Drive
# @markdown Creates a single zip containing both database files, then
# @markdown saves it to Drive. Download ONE zip file — don't download
# @markdown the individual files separately, as they may be at different
# @markdown sync states.

import shutil, time

# Verify embedding actually completed before saving
total = db.count_embeddings()
if total == 0:
    raise RuntimeError("Database has 0 embeddings — Cell 7 did not complete. "
                       "Re-run Cell 7 and wait for 'Embedding complete' before running this cell.")

print(f"Verified: {total:,} embeddings in database.")
print(f"Zipping and copying to Google Drive...")

# Create zip of both DB files
zip_base = f"/tmp/{DATASET_NAME}_db"
shutil.make_archive(zip_base, 'zip', LOCAL_DB_DIR)
zip_path = zip_base + ".zip"
zip_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"  DB zip: {zip_mb:.1f} MB")

# Save zip to Google Drive
os.makedirs(db_output_path, exist_ok=True)
dest = str(db_output_path) + f"/{DATASET_NAME}_db.zip"
shutil.copy2(zip_path, dest)
print(f"  Saved to Drive: {dest}")
print()
print("=== DOWNLOAD INSTRUCTIONS ===")
print(f"1. In Google Drive, go to: MBARI_perch/db/{DATASET_NAME}/")
print(f"2. Download: {DATASET_NAME}_db.zip")
print(f"3. On your Mac, unzip it and scp the contents to spark-ae0e:")
print()
DEST = f"/mnt/PAM_Analysis/duane_scratch/perch_hoplite/db/{DATASET_NAME}"
print(f"   ssh duane@134.89.11.107 'mkdir -p {DEST}'")
print(f"   cd ~/Downloads && unzip {DATASET_NAME}_db.zip -d {DATASET_NAME}_db/")
print(f"   scp {DATASET_NAME}_db/hoplite.sqlite duane@134.89.11.107:{DEST}/")
print(f"   scp {DATASET_NAME}_db/usearch.index  duane@134.89.11.107:{DEST}/")

Verified: 17,280 embeddings in database.
Zipping and copying to Google Drive...
  DB zip: 47.2 MB
  Saved to Drive: /content/drive/My Drive/MBARI_perch/db/MARS_20180401_20180401_32kHz/MARS_20180401_20180401_32kHz_db.zip

=== DOWNLOAD INSTRUCTIONS ===
1. In Google Drive, go to: MBARI_perch/db/MARS_20180401_20180401_32kHz/
2. Download: MARS_20180401_20180401_32kHz_db.zip
3. On your Mac, unzip it and scp the contents to spark-ae0e:

   ssh duane@134.89.11.107 'mkdir -p /mnt/PAM_Analysis/duane_scratch/perch_hoplite/db/MARS_20180401_20180401_32kHz'
   cd ~/Downloads && unzip MARS_20180401_20180401_32kHz_db.zip -d MARS_20180401_20180401_32kHz_db/
   scp MARS_20180401_20180401_32kHz_db/hoplite.sqlite duane@134.89.11.107:/mnt/PAM_Analysis/duane_scratch/perch_hoplite/db/MARS_20180401_20180401_32kHz/
   scp MARS_20180401_20180401_32kHz_db/usearch.index  duane@134.89.11.107:/mnt/PAM_Analysis/duane_scratch/perch_hoplite/db/MARS_20180401_20180401_32kHz/


In [ ]:
# @title Step 10: Transfer instructions
# @markdown Follow these steps to transfer the database to spark-ae0e.

DEST = f"/mnt/PAM_Analysis/duane_scratch/perch_hoplite/db/{DATASET_NAME}"
LABELS_DIR = "/mnt/PAM_Analysis/duane_scratch/perch_hoplite/labels"
MODELS_DIR = "/mnt/PAM_Analysis/duane_scratch/perch_hoplite/models"

print("=" * 65)
print("TRANSFER DATABASE TO SPARK-AE0E")
print("=" * 65)
print()
print("STEP 1 — Download zip from Google Drive in your browser:")
print(f"  https://drive.google.com")
print(f"  Navigate to: MBARI_perch/db/{DATASET_NAME}/")
print(f"  Download:    {DATASET_NAME}_db.zip  (~47 MB)")
print()
print("STEP 2 — On your Mac terminal, unzip:")
print(f"  cd ~/Downloads")
print(f"  unzip {DATASET_NAME}_db.zip -d {DATASET_NAME}_db/")
print(f"  ls {DATASET_NAME}_db/")
print(f"  # Should show: hoplite.sqlite  usearch.index")
print()
print("STEP 3 — Copy to spark-ae0e:")
print(f"  ssh duane@134.89.11.107 'mkdir -p {DEST}'")
print()
print(f"  scp ~/Downloads/{DATASET_NAME}_db/hoplite.sqlite \\")
print(f"      duane@134.89.11.107:{DEST}/")
print()
print(f"  scp ~/Downloads/{DATASET_NAME}_db/usearch.index \\")
print(f"      duane@134.89.11.107:{DEST}/")
print()
print("STEP 4 — Verify on spark-ae0e:")
print(f"  python3 ~/perch-hoplite/phase2_classify.py stats \\")
print(f"      --db-dir {DEST}")
print(f"  # Expecting: 17,280 embeddings")
print()
print("=" * 65)
print("AFTER VERIFICATION — Import bootstrap labels")
print("=" * 65)
print()
print("STEP 5 — Import orca bootstrap labels (April 13 2018):")
print(f"  python3 ~/perch-hoplite/phase2_classify.py label \\")
print(f"      --db-dir {DEST} \\")
print(f"      --labels-csv {LABELS_DIR}/MARS_2018_04_oo_labels.csv \\")
print(f"      --annotator-id duane")
print()
print("STEP 6 — Check label counts were imported:")
print(f"  python3 ~/perch-hoplite/phase2_classify.py stats \\")
print(f"      --db-dir {DEST}")
print()
print("=" * 65)
print("TRAIN FIRST CLASSIFIER")
print("=" * 65)
print()
print("STEP 7 — Train:")
print(f"  python3 ~/perch-hoplite/phase2_classify.py train \\")
print(f"      --db-dir {DEST} \\")
print(f"      --classifier-out {MODELS_DIR}/orca_v1.pt \\")
print(f"      --num-steps 256")
print()
print("STEP 8 — Review results with Gradio (active learning):")
print(f"  python3 ~/perch-hoplite/phase2_classify.py review \\")
print(f"      --db-dir {DEST} \\")
print(f"      --classifier {MODELS_DIR}/orca_v1.pt \\")
print(f"      --target-label orca_call \\")
print(f"      --num-results 100 \\")
print(f"      --serve --port 7860")
print(f"  # Open in browser: http://134.89.11.107:7860")
print()
print("STEP 9 — After labeling, retrain:")
print(f"  python3 ~/perch-hoplite/phase2_classify.py train \\")
print(f"      --db-dir {DEST} \\")
print(f"      --classifier-out {MODELS_DIR}/orca_v2.pt \\")
print(f"      --num-steps 256")
print()
print("  Repeat Steps 8-9 until satisfied with classifier performance.")
print()
print("STEP 10 — Full inference:")
print(f"  python3 ~/perch-hoplite/phase2_classify.py infer \\")
print(f"      --db-dir {DEST} \\")
print(f"      --classifier {MODELS_DIR}/orca_v1.pt \\")
print(f"      --output-csv {DEST}/results/orca_detections.csv \\")
print(f"      --logit-threshold 0.0")
print()
print("=" * 65)
print(f"NOTE: The full April 2018 database (4,320 files = 30 days)")
print(f"needs to be embedded on Colab as a separate run, pointing")
print(f"at the full resampled_32kHz/2018/04 directory.")
print(f"This test database covers April 13 only (144 files, 24 hours).")
print("=" * 65)


TRANSFER DATABASE TO SPARK-AE0E

STEP 1 — Download zip from Google Drive in your browser:
  https://drive.google.com
  Navigate to: MBARI_perch/db/MARS_20180401_20180401_32kHz/
  Download:    MARS_20180401_20180401_32kHz_db.zip  (~47 MB)

STEP 2 — On your Mac terminal, unzip:
  cd ~/Downloads
  unzip MARS_20180401_20180401_32kHz_db.zip -d MARS_20180401_20180401_32kHz_db/
  ls MARS_20180401_20180401_32kHz_db/
  # Should show: hoplite.sqlite  usearch.index

STEP 3 — Copy to spark-ae0e:
  ssh duane@134.89.11.107 'mkdir -p /mnt/PAM_Analysis/duane_scratch/perch_hoplite/db/MARS_20180401_20180401_32kHz'

  scp ~/Downloads/MARS_20180401_20180401_32kHz_db/hoplite.sqlite \
      duane@134.89.11.107:/mnt/PAM_Analysis/duane_scratch/perch_hoplite/db/MARS_20180401_20180401_32kHz/

  scp ~/Downloads/MARS_20180401_20180401_32kHz_db/usearch.index \
      duane@134.89.11.107:/mnt/PAM_Analysis/duane_scratch/perch_hoplite/db/MARS_20180401_20180401_32kHz/

STEP 4 — Verify on spark-ae0e:
  python3 ~/perch-h